# Singular Value Decomposition — Exercises

8 graded problems covering SVD computation, low-rank approximation, pseudoinverse, condition number, randomised SVD, LoRA, and image compression.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task |
| **Your Solution** | Code scaffold — implement the `# YOUR CODE HERE` sections |
| **Solution** | Reference implementation with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Core mechanics |
| ★★ | 4–6 | Deeper theory |
| ★★★ | 7–8 | AI applications |

### Topic Map

| Topic | Exercise |
|---|---|
| SVD computation & verification | 1, 2 |
| Eckart-Young & low-rank approx | 3 |
| Moore-Penrose pseudoinverse | 4 |
| Condition number & regularisation | 5 |
| Randomised SVD | 6 |
| LoRA low-rank adapter | 7 |
| Image compression | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la
from scipy import linalg as sla

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = [10, 5]
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '='*len(title))
    print(title)
    print('='*len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print('  expected:', np.round(expected, 6))
        print('  got     :', np.round(got, 6))
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

print('Setup complete.')

---

## Exercise 1 — SVD Computation and Verification ★

**Background.** The SVD $A = U\Sigma V^\top$ can be computed via the eigendecomposition of $A^\top A$ (for right singular vectors) and $AA^\top$ (for left singular vectors).

**Task.** Implement `compute_svd_via_eig(A)` that computes the thin SVD of $A$ using `np.linalg.eigh(A.T @ A)` (without calling `np.linalg.svd`). Then:
(a) Compute the SVD of the given matrices.
(b) Verify: $A = U\Sigma V^\top$, $A\mathbf{v}_i = \sigma_i\mathbf{u}_i$, $U^\top U = I$, $V^\top V = I$.
(c) Report the singular values and condition number of each matrix.

In [ ]:
# Exercise 1: Your Solution

def compute_svd_via_eig(A):
    """
    Compute thin SVD of A via eigendecomposition of A^T A.
    Returns (U, s, Vt) where:
      U  : (m, k) left singular vectors
      s  : (k,) singular values, descending
      Vt : (k, n) right singular vectors (V transposed)
    k = number of positive singular values
    """
    # YOUR CODE HERE
    pass


A1a = np.array([[3., 2.], [2., 3.], [0., 1.]])
A1b = np.array([[1., 0., 1.], [0., 1., 1.]])

for A in [A1a, A1b]:
    result = compute_svd_via_eig(A)
    if result is not None:
        U, s, Vt = result
        print(f'A shape={A.shape}: singular values={s.round(4)}')

In [ ]:
# Exercise 1: Solution

def compute_svd_via_eig(A):
    m, n = A.shape
    AtA = A.T @ A  # (n x n) symmetric PSD
    evals, evecs = la.eigh(AtA)  # ascending
    # Reverse to descending; keep positive only
    idx = np.argsort(evals)[::-1]
    evals = evals[idx]; evecs = evecs[:, idx]
    tol = np.finfo(float).eps * max(m, n) * evals[0] if evals[0] > 0 else 1e-10
    r = np.sum(evals > tol)
    s = np.sqrt(np.maximum(evals[:r], 0))
    V = evecs[:, :r]
    # Left singular vectors: u_i = A v_i / sigma_i
    U = (A @ V) / s[np.newaxis, :]
    return U, s, V.T


header('Exercise 1: SVD via Eigendecomposition')

for name, A in [('A1a (3x2)', np.array([[3.,2.],[2.,3.],[0.,1.]])),
                ('A1b (2x3)', np.array([[1.,0.,1.],[0.,1.,1.]]))]:
    U, s, Vt = compute_svd_via_eig(A)
    m_s, n_s = A.shape

    print(f'\n{name}: singular values = {s.round(4)}')
    check_close('U^T U = I', U.T @ U, np.eye(len(s)))
    check_close('V^T V = I', Vt @ Vt.T, np.eye(len(s)))
    check_close('A = U Sigma V^T', U @ np.diag(s) @ Vt, A)

    # Verify Av_i = sigma_i u_i
    all_ok = True
    for i in range(len(s)):
        v_i = Vt[i]; u_i = U[:, i]
        all_ok = all_ok and np.allclose(A @ v_i, s[i] * u_i)
    check_true('Av_i = sigma_i * u_i for all i', all_ok)

    # Condition number
    kappa = s[0] / s[-1] if s[-1] > 1e-10 else np.inf
    print(f'Condition number: {kappa:.2f}')

print('\nTakeaway: SVD via eigh(A^T A) works but squares the condition number — use np.linalg.svd in production.')

---

## Exercise 2 — Four Fundamental Subspaces via SVD ★

**Task.** Implement `four_subspaces(A)` that returns orthonormal bases for all four fundamental subspaces of $A$ using the SVD.

Apply it to the matrix below and verify the subspace dimensions and orthogonality:$$A = \begin{pmatrix}1&2&3\\4&5&6\\7&8&9\end{pmatrix}$$
(a) Column space, row space, null space, left null space — what are their dimensions?
(b) Verify: null space $\perp$ row space (every null space vector is orthogonal to every row space vector).
(c) Verify: $A\mathbf{v} = \mathbf{0}$ for every null space basis vector.
(d) Verify: every column of $A$ lies in the column space.

In [ ]:
# Exercise 2: Your Solution

def four_subspaces(A, tol=1e-10):
    """
    Returns dict with orthonormal bases for all four fundamental subspaces.
    Keys: 'col_space', 'left_null', 'row_space', 'null_space'
    """
    # YOUR CODE HERE
    pass


A2 = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
spaces = four_subspaces(A2)
if spaces is not None:
    for name, basis in spaces.items():
        print(f'{name}: shape={basis.shape}')

In [ ]:
# Exercise 2: Solution

def four_subspaces(A, tol=1e-10):
    m_s, n_s = A.shape
    U_s, s_s, Vt_s = la.svd(A, full_matrices=True)
    r = np.sum(s_s > tol * s_s[0])
    return {
        'col_space':  U_s[:, :r],
        'left_null':  U_s[:, r:],
        'row_space':  Vt_s[:r, :].T,
        'null_space': Vt_s[r:, :].T,
    }


A2 = np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
m2, n2 = A2.shape
r2 = la.matrix_rank(A2)
spaces = four_subspaces(A2)

header('Exercise 2: Four Fundamental Subspaces')
print(f'A: {m2}x{n2}, rank={r2}')
print(f'Expected dimensions: col={r2}, left_null={m2-r2}, row={r2}, null={n2-r2}')
for name, basis in spaces.items():
    print(f'  {name}: dim={basis.shape[1]}')

# (b) null_space perp row_space
N = spaces['null_space']
R = spaces['row_space']
overlap = N.T @ R
check_close('null_space perp row_space', overlap, np.zeros_like(overlap))

# (c) A * null_space = 0
null_test = A2 @ N
check_close('A * null_space = 0', null_test, np.zeros_like(null_test))

# (d) columns of A lie in column space
C = spaces['col_space']
for j in range(n2):
    col = A2[:, j]
    proj = C @ (C.T @ col)
    check_close(f'A[:,{j}] in col_space', proj, col)

print('\nTakeaway: SVD directly reveals the four fundamental subspaces — null space = last n-r right singular vectors.')

---

## Exercise 3 — Eckart-Young Theorem ★★

**Task.** Implement `truncated_svd(A, k)` returning the best rank-$k$ approximation $A_k$. Verify the Eckart-Young bounds and compute the minimum rank needed for 95% energy retention.

In [ ]:
# Exercise 3: Your Solution

def truncated_svd(A, k):
    """
    Returns the best rank-k approximation A_k = U_k Sigma_k Vt_k.
    """
    # YOUR CODE HERE
    pass


np.random.seed(3)
n3 = 8
target_s = np.array([10., 7., 5., 3., 2., 1., 0.5, 0.1])
Q3a, _ = la.qr(np.random.randn(n3, n3))
Q3b, _ = la.qr(np.random.randn(n3, n3))
A3 = Q3a @ np.diag(target_s) @ Q3b.T

for k in [1, 2, 4, 6]:
    Ak = truncated_svd(A3, k)
    if Ak is not None:
        err = la.norm(A3 - Ak, 'fro')
        print(f'k={k}: ||A-A_k||_F = {err:.4f}')

In [ ]:
# Exercise 3: Solution

def truncated_svd(A, k):
    U3, s3, Vt3 = la.svd(A, full_matrices=False)
    k = min(k, len(s3))
    return U3[:, :k] @ np.diag(s3[:k]) @ Vt3[:k, :]


np.random.seed(3)
n3 = 8
target_s = np.array([10., 7., 5., 3., 2., 1., 0.5, 0.1])
Q3a, _ = la.qr(np.random.randn(n3, n3))
Q3b, _ = la.qr(np.random.randn(n3, n3))
A3 = Q3a @ np.diag(target_s) @ Q3b.T
U3, s3, Vt3 = la.svd(A3, full_matrices=False)

header('Exercise 3: Eckart-Young Theorem')
print(f'Singular values: {s3.round(4)}')
print(f'\n{"k":>4} {"||A-A_k||_2":>14} {"sigma_k+1":>12} {"||A-A_k||_F":>14} {"sqrt(sum s^2,i>k)":>20}')

for k in range(1, n3):
    Ak = truncated_svd(A3, k)
    err_2 = la.norm(A3 - Ak, ord=2)
    err_F = la.norm(A3 - Ak, 'fro')
    bound_2 = s3[k]
    bound_F = np.sqrt(np.sum(s3[k:]**2))
    ok_2 = np.isclose(err_2, bound_2, rtol=1e-4)
    ok_F = np.isclose(err_F, bound_F, rtol=1e-4)
    print(f'{k:>4} {err_2:>14.6f} {bound_2:>12.6f} {err_F:>14.6f} {bound_F:>20.6f}  {"OK" if ok_2 and ok_F else "FAIL"}')

# Minimum rank for 95% energy
total_energy = np.sum(s3**2)
for k in range(1, n3+1):
    if np.sum(s3[:k]**2) / total_energy >= 0.95:
        print(f'\nMinimum rank for 95% Frobenius energy: k={k} ({100*np.sum(s3[:k]**2)/total_energy:.1f}%)')
        break

check_true('||A-A_k||_2 = sigma_{k+1} (Eckart-Young, spectral)',
           np.isclose(la.norm(A3 - truncated_svd(A3,3), ord=2), s3[3], rtol=1e-4))
print('\nTakeaway: Eckart-Young guarantees rank-k SVD is the best possible rank-k approximation in any unitarily-invariant norm.')

---

## Exercise 4 — Moore-Penrose Pseudoinverse ★★

**Task.** Implement `pseudoinverse(A, tol=None)` via SVD and verify:
(a) The four Moore-Penrose conditions.
(b) It solves overdetermined systems (least squares) and underdetermined systems (minimum norm).
(c) The geometric interpretation: $AA^+$ is the orthogonal projector onto $\text{col}(A)$.

In [ ]:
# Exercise 4: Your Solution

def pseudoinverse(A, tol=None):
    """
    Compute A^+ = V Sigma^+ U^T via SVD.
    Singular values below tol are treated as zero.
    """
    # YOUR CODE HERE
    pass


np.random.seed(4)
A4 = np.random.randn(5, 3)
Ap4 = pseudoinverse(A4)
if Ap4 is not None:
    print(f'A shape: {A4.shape}, A^+ shape: {Ap4.shape}')

In [ ]:
# Exercise 4: Solution

def pseudoinverse(A, tol=None):
    U4, s4, Vt4 = la.svd(A, full_matrices=False)
    if tol is None:
        tol = np.finfo(float).eps * max(A.shape) * s4[0]
    s_inv = np.where(s4 > tol, 1.0 / s4, 0.0)
    return Vt4.T @ np.diag(s_inv) @ U4.T


np.random.seed(4)
A4 = np.random.randn(5, 3)  # overdetermined (5x3)
Ap4 = pseudoinverse(A4)

header('Exercise 4: Moore-Penrose Pseudoinverse')
# (a) Four conditions
check_close('(1) A A^+ A = A', A4 @ Ap4 @ A4, A4)
check_close('(2) A^+ A A^+ = A^+', Ap4 @ A4 @ Ap4, Ap4)
check_close('(3) (A A^+)^T = A A^+', (A4 @ Ap4).T, A4 @ Ap4)
check_close('(4) (A^+ A)^T = A^+ A', (Ap4 @ A4).T, Ap4 @ A4)

# (b) Overdetermined: least squares
b4 = np.random.randn(5)
x_pinv = Ap4 @ b4
x_lstsq = la.lstsq(A4, b4, rcond=None)[0]
check_close('Pseudoinverse = lstsq solution', x_pinv, x_lstsq)

# Underdetermined: minimum norm
A4u = np.random.randn(3, 5)  # underdetermined (3x5)
Ap4u = pseudoinverse(A4u)
b4u = np.random.randn(3)
x_mn = Ap4u @ b4u  # minimum norm solution
# Verify it satisfies Ax = b
check_close('A x_mn = b (consistent)', A4u @ x_mn, b4u)
# Verify it's minimum norm: any other solution has larger norm
null_A4u = la.svd(A4u, full_matrices=True)[2][3:, :].T  # null space
x_other = x_mn + null_A4u[:, 0] * 0.5
check_true('||x_mn|| <= ||x_other||', la.norm(x_mn) <= la.norm(x_other) + 1e-10)

# (c) AA^+ is orthogonal projection
P_col = A4 @ Ap4
check_true('AA^+ is symmetric (orthogonal proj)', np.allclose(P_col, P_col.T))
check_true('AA^+ is idempotent (P^2=P)', np.allclose(P_col @ P_col, P_col))
print('\nTakeaway: A^+ = V Sigma^+ U^T is the unique matrix satisfying all 4 Moore-Penrose conditions.')

---

## Exercise 5 — Condition Number and Least-Squares Stability ★★

**Task.** Demonstrate how the condition number affects least-squares stability:
(a) Implement `condition_number(A)` returning $\sigma_1/\sigma_r$.
(b) Show that forming the normal equations $(A^\top A)^{-1}A^\top\mathbf{b}$ amplifies errors for ill-conditioned $A$.
(c) Implement ridge regression `ridge_solve(A, b, lam)` via spectral shrinkage.
(d) Find the optimal $\lambda$ that minimises $\|\hat{\mathbf{x}}_\lambda - \mathbf{x}^*\|$.

In [ ]:
# Exercise 5: Your Solution

def condition_number(A):
    """Return sigma_1 / sigma_r (smallest positive singular value)."""
    # YOUR CODE HERE
    pass


def ridge_solve(A, b, lam):
    """
    Solve regularised least squares: min ||Ax - b||^2 + lam * ||x||^2
    via SVD spectral shrinkage: x = V diag(s/(s^2+lam)) U^T b
    """
    # YOUR CODE HERE
    pass


np.random.seed(5)
n5 = 6
svals_5 = np.array([100., 50., 20., 5., 1., 0.01])  # ill-conditioned
print(f'Condition number of A: {svals_5[0]/svals_5[-1]:.0f}')

In [ ]:
# Exercise 5: Solution

def condition_number(A):
    s5 = la.svd(A, compute_uv=False)
    pos = s5[s5 > 1e-10 * s5[0]]
    return s5[0] / pos[-1] if len(pos) > 0 else np.inf


def ridge_solve(A, b, lam):
    U5, s5, Vt5 = la.svd(A, full_matrices=False)
    shrink = s5 / (s5**2 + lam)
    return Vt5.T @ np.diag(shrink) @ U5.T @ b


np.random.seed(5)
n5 = 6
svals_5 = np.array([100., 50., 20., 5., 1., 0.01])
Q5a, _ = la.qr(np.random.randn(n5, n5))
Q5b, _ = la.qr(np.random.randn(n5, n5))
A5 = Q5a @ np.diag(svals_5) @ Q5b.T

x_true5 = np.random.randn(n5)
b5_true = A5 @ x_true5
noise5  = 0.01 * np.random.randn(n5)
b5 = b5_true + noise5

header('Exercise 5: Condition Number & Least Squares Stability')
kappa5 = condition_number(A5)
print(f'Condition number: {kappa5:.0f}')
check_true('condition_number > 1000', kappa5 > 1000)

# (b) Normal equations vs SVD
x_svd     = la.lstsq(A5, b5, rcond=None)[0]
try:
    x_normal  = la.inv(A5.T @ A5) @ A5.T @ b5
except la.LinAlgError:
    x_normal = np.ones(n5) * np.nan

err_svd    = la.norm(x_svd    - x_true5)
err_normal = la.norm(x_normal - x_true5)
print(f'\n||x_svd    - x*||_2 = {err_svd:.4f}')
print(f'||x_normal - x*||_2 = {err_normal:.4f}')
check_true('SVD-based lstsq <= normal equations error', err_svd <= err_normal + 1e-6)

# (c-d) Ridge regression: find optimal lambda
lambdas5 = np.logspace(-4, 4, 50)
errors5  = [la.norm(ridge_solve(A5, b5, l) - x_true5) for l in lambdas5]
best_lam5 = lambdas5[np.argmin(errors5)]
best_err5 = min(errors5)
print(f'\nBest ridge lambda: {best_lam5:.4f}')
print(f'Best ridge error:  {best_err5:.4f}  (vs unregularised: {err_svd:.4f})')
check_true('Ridge improves over unregularised', best_err5 <= err_svd + 1e-4)
print('\nTakeaway: Ill-conditioning amplifies noise; ridge regression with optimal lambda significantly reduces error.')

---

## Exercise 6 — Randomised SVD ★★★

**Task.** Implement `randomized_svd(A, k, n_oversampling=5, n_iter=2)` from scratch:
(a) Draw Gaussian sketch $\Omega \in \mathbb{R}^{n \times (k+p)}$.
(b) Compute $Y = (AA^\top)^q A\Omega$ (`n_iter` power iterations).
(c) Orthogonalise $Y$ via QR.
(d) Project and compute small SVD.
(e) Verify relative error $\|A - A_k^{\text{rand}}\|_F / \|A\|_F$ is close to the exact truncated SVD error.

In [ ]:
# Exercise 6: Your Solution

def randomized_svd(A, k, n_oversampling=5, n_iter=2, random_state=0):
    """
    Randomised SVD returning (U, s, Vt) for top-k singular triplets.
    """
    # YOUR CODE HERE
    pass


np.random.seed(6)
m6, n6, rank6 = 200, 150, 10
U6, _ = la.qr(np.random.randn(m6, rank6))
V6, _ = la.qr(np.random.randn(n6, rank6))
s6_true = np.exp(-np.arange(rank6) * 0.3)
A6 = U6 @ np.diag(s6_true) @ V6.T + 0.001 * np.random.randn(m6, n6)

k6 = 8
result = randomized_svd(A6, k6)
if result is not None:
    U_r6, s_r6, Vt_r6 = result
    print(f'Returned shapes: U={U_r6.shape}, s={s_r6.shape}, Vt={Vt_r6.shape}')

In [ ]:
# Exercise 6: Solution

def randomized_svd(A, k, n_oversampling=5, n_iter=2, random_state=0):
    rng = np.random.RandomState(random_state)
    m6, n6 = A.shape
    p = k + n_oversampling
    # (a) Gaussian sketch
    Omega = rng.randn(n6, p)
    # (b) Power iterations
    Y = A @ Omega
    for _ in range(n_iter):
        Y = A @ (A.T @ Y)
    # (c) QR orthogonalise
    Q6, _ = la.qr(Y, mode='economic')
    # (d) Project and small SVD
    B6 = Q6.T @ A
    Uhat, s6, Vt6 = la.svd(B6, full_matrices=False)
    U6 = Q6 @ Uhat
    return U6[:, :k], s6[:k], Vt6[:k, :]


np.random.seed(6)
m6, n6, rank6 = 200, 150, 10
U6, _ = la.qr(np.random.randn(m6, rank6))
V6, _ = la.qr(np.random.randn(n6, rank6))
s6_true = np.exp(-np.arange(rank6) * 0.3)
A6 = U6 @ np.diag(s6_true) @ V6.T + 0.001 * np.random.randn(m6, n6)

k6 = 8
U_r6, s_r6, Vt_r6 = randomized_svd(A6, k6)
U_ex6, s_ex6, Vt_ex6 = la.svd(A6, full_matrices=False)

header('Exercise 6: Randomised SVD')
# Reconstruction errors
A_rand = U_r6 @ np.diag(s_r6) @ Vt_r6
A_exact_k = U_ex6[:, :k6] @ np.diag(s_ex6[:k6]) @ Vt_ex6[:k6, :]
frob_ref = la.norm(A6, 'fro')
err_rand  = la.norm(A6 - A_rand, 'fro') / frob_ref
err_exact = la.norm(A6 - A_exact_k, 'fro') / frob_ref

print(f'Exact rank-{k6} relative Frobenius error: {err_exact:.4f}')
print(f'Randomised rank-{k6} relative Frobenius error: {err_rand:.4f}')
check_close('Randomised singular values close to exact', s_r6, s_ex6[:k6], tol=0.01)
check_true('Randomised error < 2x exact error', err_rand < 2 * err_exact + 0.001)

# Shape checks
check_true(f'U shape = ({m6},{k6})', U_r6.shape == (m6, k6))
check_true(f's shape = ({k6},)', s_r6.shape == (k6,))
check_true(f'Vt shape = ({k6},{n6})', Vt_r6.shape == (k6, n6))
print('\nTakeaway: Randomised SVD achieves near-optimal accuracy in O(mnk) — essential for large-scale ML.')

---

## Exercise 7 — LoRA Low-Rank Adapter ★★★

**Task.** Simulate LoRA fine-tuning:
(a) Implement `lora_decompose(DeltaW, r)` that finds the rank-$r$ factorisation $B \in \mathbb{R}^{d_{out}\times r}$, $A \in \mathbb{R}^{r\times d_{in}}$ minimising $\|BA - \Delta W\|_F$ (via SVD — Eckart-Young guarantees optimality).
(b) Show that rank $r \geq r^*$ (true rank) recovers $\Delta W$ exactly.
(c) Plot the Eckart-Young error bound vs $r$.
(d) Compute the parameter savings vs the full update matrix.

In [ ]:
# Exercise 7: Your Solution

def lora_decompose(DeltaW, r):
    """
    LoRA decomposition: find B (d_out x r) and A (r x d_in) such that
    ||BA - DeltaW||_F is minimised (optimal rank-r factorisation).
    Returns (B, A)
    """
    # YOUR CODE HERE
    pass


np.random.seed(7)
d_out7, d_in7 = 64, 32
r_true7 = 4
# Construct rank-4 DeltaW
U7, _ = la.qr(np.random.randn(d_out7, r_true7))
V7, _ = la.qr(np.random.randn(d_in7, r_true7))
s7 = np.array([5., 3., 2., 1.])
DeltaW7 = U7 @ np.diag(s7) @ V7.T

B7, A7 = lora_decompose(DeltaW7, r=2) if lora_decompose(DeltaW7, 2) is not None else (None, None)
if B7 is not None:
    print(f'B shape: {B7.shape}, A shape: {A7.shape}')

In [ ]:
# Exercise 7: Solution

def lora_decompose(DeltaW, r):
    U7, s7, Vt7 = la.svd(DeltaW, full_matrices=False)
    r = min(r, len(s7))
    # B = U_r @ sqrt(Sigma_r), A = sqrt(Sigma_r) @ Vt_r
    sqrt_s = np.sqrt(s7[:r])
    B7 = U7[:, :r] @ np.diag(sqrt_s)
    A7 = np.diag(sqrt_s) @ Vt7[:r, :]
    return B7, A7


np.random.seed(7)
d_out7, d_in7 = 64, 32
r_true7 = 4
U7, _ = la.qr(np.random.randn(d_out7, r_true7))
V7, _ = la.qr(np.random.randn(d_in7, r_true7))
s7 = np.array([5., 3., 2., 1.])
DeltaW7 = U7 @ np.diag(s7) @ V7.T
_, s7_full, _ = la.svd(DeltaW7, full_matrices=False)

header('Exercise 7: LoRA Low-Rank Adapter')
print(f'DeltaW shape: {DeltaW7.shape}, true rank: {la.matrix_rank(DeltaW7)}')
print(f'True singular values: {s7_full[:8].round(4)}')
print(f'\n{"r":>4} {"Params":>10} {"Savings":>10} {"Rel error":>12} {"Exact?":>8}')

for r in [1, 2, 4, 6, 8, 16]:
    B7r, A7r = lora_decompose(DeltaW7, r)
    approx = B7r @ A7r
    err = la.norm(approx - DeltaW7, 'fro') / la.norm(DeltaW7, 'fro')
    params_lora = d_out7 * r + r * d_in7
    params_full = d_out7 * d_in7
    savings = f'{params_full/params_lora:.1f}x'
    exact = err < 1e-8
    print(f'{r:>4} {params_lora:>10} {savings:>10} {err:>12.4e} {"YES" if exact else "":>8}')

check_true('Rank-4 recovers DeltaW exactly', la.norm(lora_decompose(DeltaW7,4)[0]@lora_decompose(DeltaW7,4)[1]-DeltaW7,'fro') < 1e-8)
check_true('Rank-3 does NOT recover exactly (Eckart-Young bound > 0)',
           la.norm(lora_decompose(DeltaW7,3)[0]@lora_decompose(DeltaW7,3)[1]-DeltaW7,'fro') > 1e-4)
print('\nTakeaway: LoRA is Eckart-Young in disguise — the rank-r SVD is the parameter-efficient optimal update.')

---

## Exercise 8 — Image Compression via SVD ★★★

**Task.** Build a complete image compression pipeline:
(a) Implement `svd_compress(img, k)` returning the rank-$k$ approximation.
(b) Implement `compression_ratio(img, k)` = (original pixels) / (stored numbers in rank-$k$).
(c) Implement `psnr(original, compressed)` = $10\log_{10}(\text{MAX}^2 / \text{MSE})$.
(d) Find the rank achieving $10\times$ compression with maximum PSNR.
(e) Plot the singular value spectrum and identify the 'knee' (inflection point).

In [ ]:
# Exercise 8: Your Solution

def svd_compress(img, k):
    """Return rank-k SVD approximation of a 2D image."""
    # YOUR CODE HERE
    pass


def compression_ratio(img, k):
    """Return (m*n) / (k*(m+n+1)) — storage ratio."""
    # YOUR CODE HERE
    pass


def psnr(original, compressed, max_val=None):
    """Peak Signal-to-Noise Ratio in dB. Higher = better quality."""
    # YOUR CODE HERE
    pass


# Generate structured synthetic image
np.random.seed(8)
sz8 = 128
x8 = np.linspace(0, 4*np.pi, sz8)
img8 = (np.outer(np.sin(x8), np.cos(x8)) * 5 +
        np.outer(np.cos(2*x8), np.sin(x8)) * 3 +
        np.random.randn(sz8, sz8) * 0.3)
# Normalise to [0, 255]
img8 = (img8 - img8.min()) / (img8.max() - img8.min()) * 255.0

k_test = 10
compressed8 = svd_compress(img8, k_test)
if compressed8 is not None:
    ratio = compression_ratio(img8, k_test)
    psnr_val = psnr(img8, compressed8)
    print(f'k={k_test}: ratio={ratio:.2f}x, PSNR={psnr_val:.1f} dB')

In [ ]:
# Exercise 8: Solution

def svd_compress(img, k):
    U8, s8, Vt8 = la.svd(img, full_matrices=False)
    k = min(k, len(s8))
    return U8[:, :k] @ np.diag(s8[:k]) @ Vt8[:k, :]


def compression_ratio(img, k):
    m8, n8 = img.shape
    return (m8 * n8) / (k * (m8 + n8 + 1))


def psnr(original, compressed, max_val=None):
    if max_val is None:
        max_val = original.max()
    mse = np.mean((original - compressed)**2)
    if mse < 1e-15:
        return np.inf
    return 10 * np.log10(max_val**2 / mse)


np.random.seed(8)
sz8 = 128
x8 = np.linspace(0, 4*np.pi, sz8)
img8 = (np.outer(np.sin(x8), np.cos(x8)) * 5 +
        np.outer(np.cos(2*x8), np.sin(x8)) * 3 +
        np.random.randn(sz8, sz8) * 0.3)
img8 = (img8 - img8.min()) / (img8.max() - img8.min()) * 255.0

_, s8_full, _ = la.svd(img8, full_matrices=False)

header('Exercise 8: Image Compression via SVD')
print(f'Image: {sz8}x{sz8} = {sz8*sz8} pixels')
print(f'{"k":>6} {"Ratio":>10} {"PSNR (dB)":>12} {"% Energy":>12}')

target_ratio = 10.0
best_k_10x = None
best_psnr_10x = -np.inf

for k in [1, 5, 10, 20, 30, 50, 80, 100]:
    comp = svd_compress(img8, k)
    ratio = compression_ratio(img8, k)
    psnr_val = psnr(img8, comp)
    energy = 100 * np.sum(s8_full[:k]**2) / np.sum(s8_full**2)
    print(f'{k:>6} {ratio:>10.2f}x {psnr_val:>12.2f} {energy:>12.1f}%')
    if abs(ratio - target_ratio) < 2.0 and psnr_val > best_psnr_10x:
        best_psnr_10x = psnr_val
        best_k_10x = k

print(f'\nBest k near 10x compression: k={best_k_10x}, ratio={compression_ratio(img8, best_k_10x):.2f}x, PSNR={best_psnr_10x:.1f} dB')

# Checks
check_true('Rank-1 compression ratio > 10x', compression_ratio(img8, 1) > 10)
comp_full = svd_compress(img8, min(sz8, sz8))  # full rank
check_close('Full-rank compression = original', comp_full, img8, tol=1e-6)
check_true('PSNR increases with rank', psnr(img8, svd_compress(img8,10)) < psnr(img8, svd_compress(img8,50)))

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].semilogy(s8_full, '.')
    axes[0].set_xlabel('Index'); axes[0].set_ylabel('Singular value (log)')
    axes[0].set_title('Singular Value Spectrum')
    ks_plot = range(1, 64)
    psnrs = [psnr(img8, svd_compress(img8, k)) for k in ks_plot]
    axes[1].plot(ks_plot, psnrs, 'b-')
    axes[1].set_xlabel('Rank k'); axes[1].set_ylabel('PSNR (dB)')
    axes[1].set_title('Image PSNR vs Compression Rank')
    plt.tight_layout(); plt.show()

print('\nTakeaway: SVD gives the mathematically optimal lossy compression — Eckart-Young guarantees no rank-k method does better in the L2 sense.')

---

## What to Review After Finishing

- [ ] SVD = rotate, scale, rotate — verify $A = U\Sigma V^\top$ numerically for any matrix
- [ ] $\sigma_i = \sqrt{\lambda_i(A^\top A)}$ — never confuse with eigenvalues of $A$
- [ ] Eckart-Young: rank-$k$ SVD is the best rank-$k$ approximation (both norms)
- [ ] Pseudoinverse $A^+ = V\Sigma^+U^\top$ satisfies 4 Moore-Penrose conditions
- [ ] Condition number $\kappa = \sigma_1/\sigma_r$; ridge regression as spectral shrinkage
- [ ] Randomised SVD: sketch → power iteration → QR → small SVD; $O(mnk)$
- [ ] LoRA = Eckart-Young applied to weight update matrices

## References

1. Golub & Van Loan, *Matrix Computations*, 4th ed. — comprehensive reference for algorithms
2. Halko, Martinsson, Tropp (2011) — *Finding Structure with Randomness* — randomised SVD
3. Hu et al. (2021) — *LoRA: Low-Rank Adaptation of Large Language Models*
4. Martin & Mahoney (2021) — *Implicit Self-Regularization in Deep Neural Networks*
5. Trefethen & Bau, *Numerical Linear Algebra* — Chapters 4-6